In [54]:
import torch
import torch.nn.functional as F
from torch_geometric.nn import GCNConv
from torch_geometric.data import Data
from torch_geometric.utils import train_test_split_edges, negative_sampling
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import roc_auc_score, average_precision_score

from ast import literal_eval
import numpy as np

from torch_geometric.nn import Node2Vec

import warnings
warnings.filterwarnings('ignore')

random_seed = 62
np.random.seed(random_seed)

job_descriptions = pd.read_csv('./data/processed/job_descriptions_processed-v5.csv')
resumes = pd.read_csv('./data/processed/general-resume-dataset-processed-v6.csv', converters={'skills': literal_eval})

job_descriptions = job_descriptions.sample(frac=1, random_state=random_seed).head(20000)
print(len(job_descriptions))
print(len(resumes))
job_descriptions['skills'] = job_descriptions['skills'].apply(literal_eval)

job_descriptions['job_title'].fillna('unknown', inplace=True)
resumes['job_title'].fillna('unknown', inplace=True)
resumes['category'].fillna('unknown', inplace=True)

job_descriptions['job_id'] = range(1, len(job_descriptions) + 1)
resumes['candidate_id'] = range(1, len(resumes) + 1)

all_titles = job_descriptions['job_title'].tolist() + resumes['job_title'].tolist()
all_titles.append('unknown')
all_categories = resumes['category'].tolist()
all_categories.append('unknown')

le_job_title = LabelEncoder()
le_category = LabelEncoder()
le_job_title.fit(all_titles)
le_category.fit(all_categories)

job_descriptions['job_title'] = le_job_title.transform(job_descriptions['job_title'])
resumes['job_title'] = le_job_title.transform(resumes['job_title'])
resumes['category'] = le_category.transform(resumes['category'])

all_skills = set(skill for skills in job_descriptions['skills'].tolist() + resumes['skills'].tolist() for skill in skills)
le_skills = {skill: i for i, skill in enumerate(all_skills)}

nodes = []
edges = []
node_features = []

jobs_from_edges = []
candidates_from_edges = []
jobs_and_candidates_from_edges = []

skill_weight_multiplier = 0.25
title_weight = 0.85

for i, row in job_descriptions.iterrows():
    nodes.append(row['job_id'])
    skills_vector = [0] * len(le_skills)
    if row['skills']:
        for skill in row['skills']:
            skills_vector[le_skills[skill]] = 1
    node_features.append([row['job_title']] + skills_vector)

for i, row in resumes.iterrows():
    nodes.append(row['candidate_id'] + len(job_descriptions))
    skills_vector = [0] * len(le_skills)
    if row['skills']:
        for skill in row['skills']:
            skills_vector[le_skills[skill]] = 1
    node_features.append([row['job_title']] + skills_vector)

job_descriptions['skills'] = job_descriptions['skills'].apply(set)
resumes['skills'] = resumes['skills'].apply(set)

20000
2484


In [55]:
job_skills_dict = job_descriptions.set_index('job_id')['skills'].to_dict()
resume_skills_dict = resumes.set_index('candidate_id')['skills'].to_dict()
job_titles_dict = job_descriptions.set_index('job_id')['job_title'].to_dict()
resume_titles_dict = resumes.set_index('candidate_id')['job_title'].to_dict()

edges = []
weights = []

for job_id, job_skills in job_skills_dict.items():
    for candidate_id, resume_skills in resume_skills_dict.items():
        overlap = len(job_skills.intersection(resume_skills))
        combined_weight = overlap * skill_weight_multiplier

        if job_titles_dict[job_id] == resume_titles_dict[candidate_id]:
            combined_weight += title_weight

        if combined_weight > 0:
            edges.append((job_id, candidate_id + len(job_descriptions)))
            weights.append(combined_weight)
            jobs_and_candidates_from_edges.append((job_id, candidate_id))
            jobs_from_edges.append(job_id)
            candidates_from_edges.append(candidate_id)

nodes_length = len(nodes)

edge_index = torch.tensor(edges, dtype=torch.long).t().contiguous()
edge_weight = torch.tensor(weights, dtype=torch.float)
edge_index = edge_index.clamp(0, nodes_length - 1)

x = torch.tensor(node_features, dtype=torch.float)

data = Data(x=x, edge_index=edge_index, edge_weight=edge_weight)

original_edge_index = data.edge_index.clone()

data = train_test_split_edges(data)

edge_weight_dict = {tuple(edge_index[:, i].tolist()): edge_weight[i].item() for i in range(edge_index.size(1))}

def get_edge_weights(edge_index, edge_weight_dict):
    weights = []
    for i in range(edge_index.size(1)):
        edge = tuple(edge_index[:, i].tolist())
        weight = edge_weight_dict.get(edge, 0)  # Default to 0 if edge not found
        weights.append(weight)
    return torch.tensor(weights, dtype=torch.float)


train_edge_weights = get_edge_weights(data.train_pos_edge_index, edge_weight_dict)
test_edge_weights = get_edge_weights(data.test_pos_edge_index, edge_weight_dict)
val_edge_weights = get_edge_weights(data.val_pos_edge_index, edge_weight_dict)

data.train_pos_edge_weight = train_edge_weights
data.test_pos_edge_weight = test_edge_weights
data.val_pos_edge_weight = val_edge_weights

neg_edge_index_train = negative_sampling(
    edge_index=data.train_pos_edge_index,
    num_nodes=data.num_nodes,
    num_neg_samples=data.train_pos_edge_index.size(1),
)
data.train_neg_edge_index = neg_edge_index_train

neg_train_edge_weights = torch.zeros(neg_edge_index_train.size(1), dtype=torch.float)

neg_edge_index_test = negative_sampling(
    edge_index=data.test_pos_edge_index,
    num_nodes=data.num_nodes,
    num_neg_samples=data.test_pos_edge_index.size(1),
)
data.test_neg_edge_index = neg_edge_index_test

neg_test_edge_weights = torch.zeros(neg_edge_index_test.size(1), dtype=torch.float)

data.train_neg_edge_weight = neg_train_edge_weights

data.test_neg_edge_weight = neg_test_edge_weights

data.train_pos_edge_index = data.train_pos_edge_index.long()
data.test_pos_edge_index = data.test_pos_edge_index.long()
data.train_neg_edge_index = data.train_neg_edge_index.long()
data.test_neg_edge_index = data.test_neg_edge_index.long()

num_nodes = data.num_nodes

embedding_dim = 64
walk_length = 20
context_size = 10
walks_per_node = 10
batch_size = 128
lr = 0.01
num_epochs = 15

# Initialize Node2Vec
node2vec = Node2Vec(
    edge_index=original_edge_index,
    embedding_dim=embedding_dim,
    walk_length=walk_length,
    context_size=context_size,
    walks_per_node=walks_per_node,
    num_negative_samples=1,
    p=1,
    q=1,
    sparse=True,
    num_nodes=num_nodes  # Specify number of nodes
)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
node2vec = node2vec.to(device)

optimizer = torch.optim.SparseAdam(node2vec.parameters(), lr=lr)

def train_node2vec(num_epochs):
    node2vec.train()
    for epoch in range(num_epochs):
        total_loss = 0
        loader = node2vec.loader(batch_size=batch_size, shuffle=True, num_workers=0)  # Set num_workers=0
        for i, (pos_rw, neg_rw) in enumerate(loader):
            optimizer.zero_grad()
            loss = node2vec.loss(pos_rw.to(device), neg_rw.to(device))
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
            if epoch % 10 == 0:
                print(f'Epoch {epoch + 1}, Iteration {i}, Loss: {total_loss / 10}')
                total_loss = 0

train_node2vec(num_epochs)

node_embeddings = node2vec.embedding.weight.data.cpu().numpy()
data.x = torch.tensor(node_embeddings, dtype=torch.float)

class GAE(torch.nn.Module):
    def __init__(self, in_channels, out_channels):
        super(GAE, self).__init__()
        self.conv1 = GCNConv(in_channels, 2 * out_channels)
        self.conv2 = GCNConv(2 * out_channels, out_channels)

    def encode(self, x, edge_index, edge_weight):
        x = F.relu(self.conv1(x, edge_index, edge_weight))
        return self.conv2(x, edge_index, edge_weight)

    def decode(self, z, pos_edge_index, neg_edge_index):
        pos_pred = (z[pos_edge_index[0].long()] * z[pos_edge_index[1].long()]).sum(dim=1)
        neg_pred = (z[neg_edge_index[0].long()] * z[neg_edge_index[1].long()]).sum(dim=1)
        return pos_pred, neg_pred

    def forward(self, data):
        z = self.encode(data.x, data.train_pos_edge_index, data.train_pos_edge_weight)
        return z

Epoch 1, Iteration 0, Loss: 0.3802082061767578
Epoch 1, Iteration 1, Loss: 0.3808866262435913
Epoch 1, Iteration 2, Loss: 0.37416558265686034
Epoch 1, Iteration 3, Loss: 0.3726069450378418
Epoch 1, Iteration 4, Loss: 0.36915485858917235
Epoch 1, Iteration 5, Loss: 0.36863417625427247
Epoch 1, Iteration 6, Loss: 0.37339613437652586
Epoch 1, Iteration 7, Loss: 0.3685302734375
Epoch 1, Iteration 8, Loss: 0.366489052772522
Epoch 1, Iteration 9, Loss: 0.36685843467712403
Epoch 1, Iteration 10, Loss: 0.3664984226226807
Epoch 1, Iteration 11, Loss: 0.36451475620269774
Epoch 1, Iteration 12, Loss: 0.3678034782409668
Epoch 1, Iteration 13, Loss: 0.3639819622039795
Epoch 1, Iteration 14, Loss: 0.3593138217926025
Epoch 1, Iteration 15, Loss: 0.36208376884460447
Epoch 1, Iteration 16, Loss: 0.3602685213088989
Epoch 1, Iteration 17, Loss: 0.3572566986083984
Epoch 1, Iteration 18, Loss: 0.3604928255081177
Epoch 1, Iteration 19, Loss: 0.35726478099823
Epoch 1, Iteration 20, Loss: 0.3545744180679321
E

In [56]:
print(data.num_node_features)

64


In [57]:
model = GAE(data.num_node_features, 16)
gae_optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
loss_fn = torch.nn.BCEWithLogitsLoss()

def train(data):
    model.train()
    gae_optimizer.zero_grad()
    z = model.encode(data.x, data.train_pos_edge_index, data.train_pos_edge_weight)  # Pass train_pos_edge_weight
    pos_pred, neg_pred = model.decode(z, data.train_pos_edge_index, data.train_neg_edge_index)
    pos_loss = loss_fn(pos_pred, torch.ones_like(pos_pred))
    neg_loss = loss_fn(neg_pred, torch.zeros_like(neg_pred))
    loss = pos_loss + neg_loss
    if torch.isnan(loss) or torch.isinf(loss):
        print("Warning: NaN or Inf loss detected")
        return float('inf')
    loss.backward()
    gae_optimizer.step()
    return loss.item()

for epoch in range(500):
    loss = train(data)
    if loss == float('inf'):
        break
    if epoch % 10 == 0:
        print(f'Epoch {epoch}, Loss: {loss}')

def average_precision(y_true, y_pred):
    idx = np.argsort(y_pred)[::-1]
    y_true_sorted = y_true[idx]
    tp = np.cumsum(y_true_sorted)
    precision = tp / (np.arange(len(y_true_sorted)) + 1)
    avg_precision = np.sum(precision * y_true_sorted) / np.sum(y_true_sorted)
    return avg_precision

def mean_average_precision(y_true, y_pred):
    return np.mean([average_precision(y_t, y_p) for y_t, y_p in zip(y_true, y_pred)])

def evaluate_model(data, model, k):
    model.eval()
    with torch.no_grad():
        z = model.encode(data.x, data.val_pos_edge_index, data.val_pos_edge_weight)
        pos_pred = torch.sigmoid((z[data.val_pos_edge_index[0].long()] * z[data.val_pos_edge_index[1].long()]).sum(dim=1)).cpu().numpy()
        neg_pred = torch.sigmoid((z[data.val_neg_edge_index[0].long()] * z[data.val_neg_edge_index[1].long()]).sum(dim=1)).cpu().numpy()

    y_true = np.concatenate([np.ones(pos_pred.shape[0]), np.zeros(neg_pred.shape[0])])
    y_pred = np.concatenate([pos_pred, neg_pred])

    auc_roc = roc_auc_score(y_true, y_pred)
    ap = average_precision_score(y_true, y_pred)
    map_score = mean_average_precision([y_true], [y_pred])

    return auc_roc, ap, map_score

k=10

auc_roc, ap, map_score = evaluate_model(data, model, k)
print(f"AUC-ROC: {auc_roc:.4f}")
print(f"AP: {ap:.4f}")
print(f"MAP: {map_score:.4f}")

def predict_best_candidates(job_descriptions, resumes, z, k=1):
    job_ids = job_descriptions['job_id'].values
    candidate_ids = resumes['candidate_id'].values + len(job_descriptions)

    job_indices = job_ids - 1
    candidate_indices = candidate_ids - 1

    job_embeddings = z[job_indices]
    candidate_embeddings = z[candidate_indices]

    # Calculate scores using matrix multiplication
    scores = torch.sigmoid(torch.matmul(job_embeddings, candidate_embeddings.T)).cpu().numpy()

    predictions = []
    for i, job_id in enumerate(job_ids):
        best_match_indices = scores[i].argsort()[::-1][:k]
        for idx in best_match_indices:
            candidate_id = resumes.iloc[idx]['candidate_id']
            candidate_job_title = resumes.iloc[idx]['job_title']
            candidate_recommendation_id = resumes.iloc[idx]['recommendation_id']
            category = resumes.iloc[idx]['category']
            skills = resumes.iloc[idx]['skills']
            job_title = job_descriptions.iloc[i]['job_title']
            job_skills = job_descriptions.iloc[i]['skills']
            score = scores[i][idx]

            match_percentage = score * 100  # Assuming the score is between 0 and 1
            predictions.append({
                "Job ID": job_id,
                "Job Title": le_job_title.inverse_transform([job_title])[0],
                "Candidate ID": candidate_id,
                "Candidate Recommendation ID": candidate_recommendation_id,
                "Candidate Job Title": le_job_title.inverse_transform([candidate_job_title])[0],
                "Candidate Category": le_category.inverse_transform([category])[0],
                "Match Percentage": match_percentage,
                "Mutual Skills": set(job_skills).intersection(set(skills)),
                "Job Skills": job_skills,
                "Candidate Skills": skills
            })

    predictions_df = pd.DataFrame(predictions)
    return predictions_df

# Example usage
with torch.no_grad():
    z = model.encode(data.x, data.test_pos_edge_index, data.test_pos_edge_weight)

predictions_df = predict_best_candidates(job_descriptions, resumes, z)
predictions_df = predictions_df[predictions_df['Mutual Skills'].map(len) != 0]
predictions_df['Mutual Skills Count'] = predictions_df['Mutual Skills'].map(len)
ind = predictions_df['Mutual Skills'].map(len).sort_values(ascending=False).index
predictions_df = predictions_df.reindex(ind)

Epoch 0, Loss: 14.52565860748291
AUC-ROC: 0.8849
AP: 0.8504
MAP: 0.8570


In [58]:
predictions_df.head(10000)

,Job ID,Job Title,Candidate ID,Candidate Recommendation ID,Candidate Job Title,Candidate Category,Match Percentage,Mutual Skills,Job Skills,Candidate Skills,Mutual Skills Count
10021,10022,customer success manager,238,2e7be612-3d08-4814-ae24-fe37e314cca1,senior commercial business banking relationshi...,banking,100.000000,"{business development, relationship management...","{business development, account management, cus...","{treasury management, sale presentation, retai...",3
12003,12004,operations manager,238,2e7be612-3d08-4814-ae24-fe37e314cca1,senior commercial business banking relationshi...,banking,100.000000,"{strategic planning, operation management, pro...","{team management, strategic planning, financia...","{treasury management, sale presentation, retai...",3
11985,11986,procurement manager,2334,4d96a4cd-307a-4626-b80b-839da9c8079b,staffing business development manager,business-development,100.000000,"{contract negotiation, relationship management...","{financial acumen, market research, supplier r...","{business development, market trend, sale reco...",3
15640,15641,marketing manager,2126,a5f4c5b1-94cf-41fc-8173-72eeee2f014c,product marketing manager,apparel,100.000000,"{competitive analysis, pricing strategy, marke...","{competitive analysis, pricing strategy, marke...","{target market, market research, competitive a...",3
14248,14249,purchasing agent,1373,260bc7cc-477d-4419-a402-9304d650a339,sales and business development,business-development,100.000000,"{supply chain management, supply chain, invent...","{inventory control, supply chain management, d...","{customer experience, e business, crm, busines...",3
...,...,...,...,...,...,...,...,...,...,...,...
6906,6907,digital marketing specialist,1388,1f0f2405-f95a-4b0c-9c29-96f234a589ba,marketing manager,banking,99.998868,{social medium},"{social medium analytic, instagram content, so...","{secondary market, e commerce, crm, com, due d...",1
6904,6905,customer support specialist,959,ae1ec72a-c8a2-4305-ad67-440e9bcbd502,teacher,teacher,99.999988,{customer service},"{knowledge base, customer service, help desk s...","{vocal music, quality control, customer servic...",1
6894,6895,investment banker,2262,8a9993a4-4047-46ea-b38f-c5a2a18e03dd,senior compliance officer,banking,100.000000,{due diligence},"{due diligence, merger and acquisition}","{write communication, retail banking, service ...",1
6886,6887,systems administrator,2484,51e45b8f-23f0-49cb-a0a2-08417a3c5ded,office manager,fitness,100.000000,{customer service},"{customer service, problem solve}","{financial accounting, federal law, proprietar...",1


In [61]:
def predict_for_new_job_v1(new_job_desc, model, data, le_job_title, le_skills, le_category, resumes, job_descriptions,
                        skill_weight=0.25, title_weight=0.85, k=5):
    """
    Predict top candidates for a new job description using normalized similarity scores.

    Args:
        new_job_desc (dict): New job description with 'job_title' and 'skills'.
        model (GAE): Trained GAE model.
        data (Data): PyTorch Geometric Data object.
        le_job_title (LabelEncoder): Encoder for job titles.
        le_skills (dict): Encoder for skills.
        le_category (LabelEncoder): Encoder for categories.
        resumes (DataFrame): Candidate data.
        job_descriptions (DataFrame): Job data.
        skill_weight (float): Weight for normalized skill overlap.
        title_weight (float): Weight for title match.
        k (int): Number of top candidates to return.

    Returns:
        DataFrame: Top-k candidates with match details.
    """
    # Encode job title and skills
    try:
        job_title = le_job_title.transform([new_job_desc['job_title']])[0]
    except ValueError:
        job_title = le_job_title.transform(['unknown'])[0]

    skills_vector = [0] * len(le_skills)
    for skill in new_job_desc['skills']:
        if skill in le_skills:
            skills_vector[le_skills[skill]] = 1

    new_job_features = torch.tensor([job_title] + skills_vector, dtype=torch.float).unsqueeze(0)

    # Project or truncate the new job features to match data.x dimensions
    if new_job_features.size(1) > data.x.size(1):  # Truncate if too large
        new_job_features_projected = new_job_features[:, :data.x.size(1)]
    else:  # Pad if too small
        padding = torch.zeros((1, data.x.size(1) - new_job_features.size(1)))
        new_job_features_projected = torch.cat([new_job_features, padding], dim=1)

    new_job_features_projected = new_job_features_projected.to(data.x.device)

    # Append the new node to the feature matrix
    updated_x = torch.cat([data.x, new_job_features_projected], dim=0)

    # Encode the entire graph, including the new job node
    new_node_index = data.x.size(0)
    model.eval()
    with torch.no_grad():
        z = model.encode(updated_x, data.val_pos_edge_index, data.val_pos_edge_weight)
        new_job_embedding = z[new_node_index].unsqueeze(0)
        candidate_embeddings = z[len(data.x) - len(resumes):]

    # Compute similarity scores
    raw_scores = torch.matmul(new_job_embedding, candidate_embeddings.T).cpu().numpy().flatten()

    # Adjust and normalize scores
    predictions = []
    for idx, candidate in resumes.iterrows():
        candidate_id = candidate['candidate_id']
        candidate_job_title = candidate['job_title']
        candidate_skills = candidate['skills']
        candidate_recommendation_id = candidate['recommendation_id']

        # Skill overlap normalization
        job_skills = set(new_job_desc['skills'])
        mutual_skills = job_skills.intersection(candidate_skills)
        normalized_skill_score = len(mutual_skills) / max(len(job_skills), len(candidate_skills))

        # Title similarity
        title_score = 1 if job_title == candidate_job_title else 0

        # Final score calculation
        final_score = raw_scores[idx] * ((normalized_skill_score * skill_weight) + (title_score * title_weight))

        predictions.append({
            "Job ID": "New Job",
            "Job Title": le_job_title.inverse_transform([job_title])[0],
            "Candidate ID": candidate_id,
            "Candidate Recommendation ID": candidate_recommendation_id,
            "Candidate Job Title": le_job_title.inverse_transform([candidate_job_title])[0],
            # "Match Percentage": final_score * 100,  # Convert to percentage
            "OG Score": raw_scores[idx],
            "Match Score": final_score,
            "Mutual Skills": mutual_skills,
            "Job Skills": list(job_skills),
            "Candidate Skills": candidate_skills
        })

    # Convert to DataFrame and sort
    predictions_df = pd.DataFrame(predictions)
    return predictions_df.sort_values(by="Match Score", ascending=False).head(k)

In [62]:
# New job description
new_job = {
    "job_title": "hr coordinator",
    "skills": ["employee handbook", "development management", "problem solve", "state law", "resource management", "human resource management", "benefit administration", "microsoft office", "customer service", "leadership development"]
}

# Predict top candidates
top_candidates = predict_for_new_job_v1(
    new_job_desc=new_job,
    model=model,
    data=data,
    le_job_title=le_job_title,
    le_skills=le_skills,
    le_category=le_category,
    resumes=resumes,
    job_descriptions=job_descriptions,
    k=5
)

# Display the top candidates
top_candidates.head(10000)


,Job ID,Job Title,Candidate ID,Candidate Recommendation ID,Candidate Job Title,OG Score,Match Score,Mutual Skills,Job Skills,Candidate Skills
400,New Job,hr coordinator,401,26c6e8dc-bde0-4ccb-8256-4287fbadaa17,hr coordinator,2307.138672,2249.460205,"{state law, benefit administration, developmen...","[state law, benefit administration, developmen...","{development management, employee handbook, or..."
93,New Job,hr coordinator,94,07fb19db-1748-4421-bf24-fcdf9c44ea70,hr coordinator,2177.542969,1878.130811,{customer service},"[state law, benefit administration, developmen...","{financial accounting, disciplinary procedure,..."
2433,New Job,hr coordinator,2434,1da1d2e4-b541-4f6e-b362-9fa86cd9be2e,hr coordinator,883.829041,778.874342,"{state law, customer service}","[state law, benefit administration, developmen...","{osha, state law, social medium, adp, data ent..."
917,New Job,hr coordinator,918,4cc62b61-4f17-48e8-aa09-7bbd8dd0e0b2,hr coordinator,695.033325,615.600945,"{microsoft office, state law, problem solve}","[state law, benefit administration, developmen...","{goal orient, persuasive communication, employ..."
378,New Job,hr coordinator,379,02e07a86-d7b4-4472-ab7c-e8c3344273df,hr coordinator,635.612793,554.716619,{microsoft office},"[state law, benefit administration, developmen...","{expense report, ibm, applicant tracking syste..."


In [ ]:
import os
import torch
import pickle

# Define the folder for saving the model
model_folder = "gae-model-v1"
os.makedirs(model_folder, exist_ok=True)

# Save the GAE model
model_path = os.path.join(model_folder, "model.pth")
torch.save(model.state_dict(), model_path)
print(f"Model saved to {model_path}")

# Save the graph data object
data_path = os.path.join(model_folder, "graph_data.pkl")
with open(data_path, "wb") as f:
    pickle.dump(data, f)
print(f"Graph data saved to {data_path}")

# Save the encoders and metadata
encoders_path = os.path.join(model_folder, "encoders_metadata.pkl")
encoders_metadata = {
    "le_job_title": le_job_title,
    "le_skills": le_skills,
    "le_category": le_category,
    "num_node_features": data.num_node_features  # Include num_node_features
}
with open(encoders_path, "wb") as f:
    pickle.dump(encoders_metadata, f)
print(f"Encoders and metadata saved to {encoders_path}")